# Bài 08 - Mẫu Thiết Kế Đa Tác Nhân


## Thiết lập


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tại sao lại dùng Hệ thống Đa Tác nhân?

Các nhiệm vụ thực tế như lập kế hoạch chuyến đi đòi hỏi nhiều loại chuyên môn khác nhau — như hậu cần, kiến thức địa phương, ngân sách, và hơn thế nữa. Một tác nhân đơn lẻ cố gắng xử lý mọi thứ sẽ nhanh chóng trở nên khó kiểm soát.

Hệ thống đa tác nhân giải quyết điều này thông qua **chuyên môn hóa**: mỗi tác nhân tập trung vào một lĩnh vực chuyên môn, tạo ra kết quả chất lượng cao hơn so với một người làm tất cả. Chúng cũng cải thiện **khả năng mở rộng** — bạn có thể thêm các tác nhân mới (ví dụ: chuyên gia chuyến bay, nhà phê bình nhà hàng) mà không cần viết lại luồng công việc hiện có. Các tác nhân kết hợp với nhau thông qua một quy trình có cấu trúc, truyền bối cảnh từ người này sang người tiếp theo.


## Tạo các Đại lý Chuyên biệt


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Xây dựng một Quy trình Làm việc Tuần tự

`WorkflowBuilder` cho phép bạn kết nối các tác nhân thành một đồ thị có hướng. Ở đây chúng ta tạo một quy trình hai bước đơn giản: **TravelPlanner** soạn thảo hành trình, sau đó **TravelConcierge** xem xét và cải thiện nó.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Thêm Nhiều Đại Lý Vào Quy Trình Làm Việc

Một trong những lợi thế lớn nhất của mô hình đa đại lý là sự dễ dàng mở rộng. Dưới đây chúng ta thêm một đại lý **BudgetReviewer** để kiểm tra kế hoạch so với ngân sách của người đi du lịch, đánh dấu các mục có thể đẩy chi phí vượt quá giới hạn và đề xuất các lựa chọn tiết kiệm chi phí. Quy trình làm việc hiện tại chạy ba đại lý theo trình tự:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Tóm tắt

Trong bài học này bạn đã học cách:

1. **Tạo các tác nhân chuyên biệt** — mỗi tác nhân có một vai trò tập trung (lập kế hoạch, quản gia, xem xét ngân sách).
2. **Kết nối các tác nhân vào một quy trình làm việc theo trình tự** sử dụng `WorkflowBuilder` và `add_edge`.
3. **Phát trực tiếp đầu ra** từ một chuỗi các tác nhân đa dạng, theo dõi tác nhân nào đang nói.
4. **Mở rộng quy trình làm việc** bằng cách thêm các tác nhân mới vào chuỗi mà không làm thay đổi các tác nhân hiện có.

Mẫu thiết kế đa tác nhân giữ mỗi tác nhân đơn giản trong khi tạo ra kết quả phong phú và được đánh giá kỹ lưỡng hơn so với bất kỳ tác nhân đơn lẻ nào có thể đạt được một mình.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
